In [ ]:
# Install all required packages
import subprocess, sys

packages = [
    'scapy',
    'scikit-learn',
    'numpy',
    'pandas',
    'torch',          # CPU-only torch is fine
    'torchvision',
    'tqdm',
    'matplotlib',
    'seaborn',
    'py7zr',          # for extracting .7z files
    'networkx',       # for GraphDApp graph construction
    'transformers',   # for PERT / ET-BERT
    'datasets',
    'tabulate',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('All dependencies installed.')

In [ ]:
import py7zr, os, glob

DATA_ROOT = 'data'   # <-- Change this to your actual data folder path

archives = glob.glob(os.path.join(DATA_ROOT, '**', '*.7z'), recursive=True)
if not archives:
    print('No .7z files found — skipping extraction.')
else:
    for arc in archives:
        dest = os.path.dirname(arc)
        print(f'Extracting {arc} -> {dest}')
        with py7zr.SevenZipFile(arc, mode='r') as z:
            z.extractall(path=dest)
    print('Done.')

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
from scapy.all import rdpcap, IP, TCP, UDP
from collections import defaultdict
from tqdm import tqdm

warnings.filterwarnings('ignore')

DATA_ROOT = 'data'   # <-- set your path here
MAX_FLOWS_PER_CLASS = 500   # keep manageable on CPU; set higher if you have time
MAX_PACKETS_PER_FLOW = 20
MAX_PAYLOAD_BYTES = 784     # 28x28 for Deeppacket

# ─── Helpers ──────────────────────────────────────────────────────────────────

def flow_key(pkt):
    """5-tuple flow key (bidirectional)."""
    if IP in pkt:
        proto = pkt[IP].proto
        src, dst = pkt[IP].src, pkt[IP].dst
        sport = pkt.sport if hasattr(pkt, 'sport') else 0
        dport = pkt.dport if hasattr(pkt, 'dport') else 0
        key = tuple(sorted([(src, sport), (dst, dport)])) + (proto,)
        return key
    return None

def extract_flows(pcap_path, max_flows=MAX_FLOWS_PER_CLASS):
    """Group packets into flows; return list of packet-length sequences."""
    try:
        pkts = rdpcap(pcap_path)
    except Exception as e:
        print(f'  [WARN] Could not read {pcap_path}: {e}')
        return []

    flows = defaultdict(list)
    for pkt in pkts:
        k = flow_key(pkt)
        if k:
            flows[k].append(pkt)

    result = []
    for pkts_in_flow in list(flows.values())[:max_flows]:
        lengths = [len(p) for p in pkts_in_flow[:MAX_PACKETS_PER_FLOW]]
        raw_bytes = b''.join(bytes(p)[:MAX_PAYLOAD_BYTES // MAX_PACKETS_PER_FLOW]
                             for p in pkts_in_flow[:MAX_PACKETS_PER_FLOW])
        result.append({'lengths': lengths, 'raw': raw_bytes, 'n_pkts': len(pkts_in_flow)})
    return result

# ─── Statistical features (AppScanner / CUMUL / BIND / K-FP) ────────────────

def stat_features(lengths):
    """54 statistical features similar to AppScanner."""
    a = np.array(lengths, dtype=float)
    if len(a) == 0:
        return np.zeros(54)
    feats = []
    # basic stats
    feats += [a.mean(), a.std(), a.min(), a.max(),
              np.percentile(a, 25), np.percentile(a, 50), np.percentile(a, 75)]
    # counts
    feats += [len(a), np.sum(a), np.sum(a > 0), np.sum(a < 0)]
    # inter-packet deltas
    deltas = np.diff(a) if len(a) > 1 else np.array([0.])
    feats += [deltas.mean(), deltas.std(), deltas.min(), deltas.max()]
    # cumulative sum features (CUMUL-style) — 20 equidistant points
    cumsum = np.cumsum(np.abs(a))
    idx = np.linspace(0, len(cumsum) - 1, 20).astype(int)
    feats += list(cumsum[idx])
    # burst features (BIND-style) — split into 10 bins
    bins = np.array_split(a, 10)
    feats += [b.sum() if len(b) else 0. for b in bins]
    # zero-pad / truncate to exactly 54
    feats = feats[:54]
    feats += [0.] * (54 - len(feats))
    return np.array(feats, dtype=np.float32)

def cumul_features(lengths, n_features=100):
    """CUMUL-style cumulative features."""
    a = np.array(lengths, dtype=float)
    if len(a) == 0:
        return np.zeros(n_features + 4)
    cumsum = np.cumsum(np.abs(a))
    idx = np.linspace(0, len(cumsum) - 1, n_features).astype(int)
    base = [len(a), cumsum[-1], np.sum(a > 0), np.sum(a < 0)]
    return np.array(base + list(cumsum[idx]), dtype=np.float32)

# ─── Sequence features (FS-Net / TSCRNN) ──────────────────────────────────────

def pad_sequence(lengths, maxlen=MAX_PACKETS_PER_FLOW):
    a = np.array(lengths[:maxlen], dtype=np.float32)
    if len(a) < maxlen:
        a = np.pad(a, (0, maxlen - len(a)))
    return a

# ─── Raw-byte features (Deeppacket) ───────────────────────────────────────────

def raw_features(raw_bytes, size=MAX_PAYLOAD_BYTES):
    arr = np.frombuffer(raw_bytes[:size], dtype=np.uint8).astype(np.float32) / 255.0
    if len(arr) < size:
        arr = np.pad(arr, (0, size - len(arr)))
    return arr

# ─── Load all data ─────────────────────────────────────────────────────────────

LABEL_MAP = {}
all_stat, all_cumul, all_seq, all_raw, all_labels = [], [], [], [], []

for split in ['Benign', 'Malware']:
    folder = os.path.join(DATA_ROOT, split)
    if not os.path.isdir(folder):
        print(f'[WARN] Folder not found: {folder}'); continue
    pcaps = sorted(glob.glob(os.path.join(folder, '*.pcap')))
    for pcap in pcaps:
        label_name = os.path.splitext(os.path.basename(pcap))[0]
        if label_name not in LABEL_MAP:
            LABEL_MAP[label_name] = len(LABEL_MAP)
        label_id = LABEL_MAP[label_name]
        print(f'  Loading {pcap} (label={label_name})')
        flows = extract_flows(pcap)
        for f in flows:
            all_stat.append(stat_features(f['lengths']))
            all_cumul.append(cumul_features(f['lengths']))
            all_seq.append(pad_sequence(f['lengths']))
            all_raw.append(raw_features(f['raw']))
            all_labels.append(label_id)

X_stat   = np.array(all_stat)
X_cumul  = np.array(all_cumul)
X_seq    = np.array(all_seq)
X_raw    = np.array(all_raw)
y        = np.array(all_labels)

print(f'\nTotal flows: {len(y)}')
print(f'Classes ({len(LABEL_MAP)}): {list(LABEL_MAP.keys())}')
print(f'X_stat shape:  {X_stat.shape}')
print(f'X_cumul shape: {X_cumul.shape}')
print(f'X_seq shape:   {X_seq.shape}')
print(f'X_raw shape:   {X_raw.shape}')

In [ ]:
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from tabulate import tabulate

os.makedirs('results', exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

def split(X):
    return train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

# Pre-split all feature sets
Xstat_tr, Xstat_te, y_tr, y_te = split(X_stat)
Xcumul_tr, Xcumul_te, _, _     = split(X_cumul)
Xseq_tr, Xseq_te, _, _         = split(X_seq)
Xraw_tr, Xraw_te, _, _         = split(X_raw)

# Scale stat/cumul features
scaler_stat = StandardScaler().fit(Xstat_tr)
Xstat_tr_s = scaler_stat.transform(Xstat_tr)
Xstat_te_s = scaler_stat.transform(Xstat_te)

scaler_cumul = StandardScaler().fit(Xcumul_tr)
Xcumul_tr_s = scaler_cumul.transform(Xcumul_tr)
Xcumul_te_s = scaler_cumul.transform(Xcumul_te)

RESULTS = {}   # model_name -> {PR, RC, F1, Acc}

def evaluate(name, y_true, y_pred):
    pr  = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rc  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    RESULTS[name] = {'PR': round(pr,4), 'RC': round(rc,4), 'F1': round(f1,4), 'Acc': round(acc,4)}
    print(f'[{name}]  PR={pr:.4f}  RC={rc:.4f}  F1={f1:.4f}  Acc={acc:.4f}')
    return RESULTS[name]

print('Train size:', len(y_tr), '  Test size:', len(y_te))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

print('Training AppScanner (Random Forest, 54 statistical features)...')
appscanner = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE)
appscanner.fit(Xstat_tr_s, y_tr)
pred = appscanner.predict(Xstat_te_s)
evaluate('AppScanner', y_te, pred)

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

print('Training CUMUL (LinearSVC on cumulative packet-size features)...')
cumul_model = LinearSVC(C=1.0, max_iter=2000, random_state=RANDOM_STATE)
cumul_model.fit(Xcumul_tr_s, y_tr)
pred = cumul_model.predict(Xcumul_te_s)
evaluate('CUMUL', y_te, pred)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# BIND uses bi-directional burst features; we approximate with stat features
# For a CPU-friendly run we use n_estimators=50
print('Training BIND (GradientBoosting on burst features)...')
bind_model = GradientBoostingClassifier(n_estimators=50, max_depth=4,
                                        learning_rate=0.1, random_state=RANDOM_STATE)
bind_model.fit(Xstat_tr_s, y_tr)
pred = bind_model.predict(Xstat_te_s)
evaluate('BIND', y_te, pred)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

print('Training K-FP (Random Forest leaf indices → k-NN)...')

# Step 1: fit RF, extract leaf indices as new feature space
rf_kfp = RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=RANDOM_STATE)
rf_kfp.fit(Xstat_tr_s, y_tr)

leaf_tr = rf_kfp.apply(Xstat_tr_s)   # shape (n, n_trees)
leaf_te = rf_kfp.apply(Xstat_te_s)

# Step 2: train k-NN on the leaf representation
kfp_model = KNeighborsClassifier(n_neighbors=5, n_jobs=-1, metric='hamming')
kfp_model.fit(leaf_tr, y_tr)
pred = kfp_model.predict(leaf_te)
evaluate('K-FP', y_te, pred)

In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import normalize

# FlowPrint: cluster flows into fingerprints, then assign labels by majority vote
# We implement the core idea: cluster training flows, label clusters, predict via NN
print('Training FlowPrint (semi-supervised clustering fingerprinting)...')

n_clusters = min(len(LABEL_MAP) * 5, len(Xstat_tr_s) // 5)
kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=5)
cluster_ids = kmeans.fit_predict(Xstat_tr_s)

# Label each cluster by majority vote
cluster_label = {}
for c in range(n_clusters):
    mask = cluster_ids == c
    if mask.sum() > 0:
        vals, counts = np.unique(y_tr[mask], return_counts=True)
        cluster_label[c] = vals[counts.argmax()]
    else:
        cluster_label[c] = 0

# Predict: assign each test flow to nearest cluster
te_clusters = kmeans.predict(Xstat_te_s)
pred = np.array([cluster_label.get(c, 0) for c in te_clusters])
evaluate('FlowPrint', y_te, pred)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class FSNet(nn.Module):
    """Lightweight GRU-based flow sequence network (FS-Net)."""
    def __init__(self, input_size=1, hidden=64, n_layers=2, n_classes=20):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden, n_layers, batch_first=True, dropout=0.3)
        self.fc  = nn.Linear(hidden, n_classes)

    def forward(self, x):           # x: (B, T)
        x = x.unsqueeze(-1)         # (B, T, 1)
        _, h = self.gru(x)
        return self.fc(h[-1])


def train_nn(model, X_tr, y_tr, X_te, y_te,
             epochs=15, batch_size=64, lr=1e-3, device='cpu'):
    model.to(device)
    loader = DataLoader(TensorDataset(torch.FloatTensor(X_tr),
                                      torch.LongTensor(y_tr)),
                        batch_size=batch_size, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        model.train(); total_loss = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
            total_loss += loss.item()
        if epoch % 5 == 0:
            print(f'  Epoch {epoch}/{epochs}  loss={total_loss/len(loader):.4f}')

    model.eval()
    with torch.no_grad():
        logits = model(torch.FloatTensor(X_te).to(device))
    return logits.argmax(1).cpu().numpy()


N_CLASSES = len(LABEL_MAP)

print(f'Training FS-Net (GRU, {N_CLASSES} classes)...')
fsnet = FSNet(n_classes=N_CLASSES)
pred = train_nn(fsnet, Xseq_tr, y_tr, Xseq_te, y_te)
evaluate('FS-Net', y_te, pred)

In [ ]:
class DeepFingerprinting(nn.Module):
    """1D CNN for website fingerprinting (Deep Fingerprinting)."""
    def __init__(self, input_len=MAX_PACKETS_PER_FLOW, n_classes=20):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1), nn.ELU(), nn.Dropout(0.1),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=3, padding=1), nn.ELU(), nn.Dropout(0.1),
            nn.AdaptiveAvgPool1d(4),
        )
        self.fc = nn.Sequential(
            nn.Linear(64 * 4, 128), nn.ELU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):           # x: (B, T)
        x = x.unsqueeze(1)          # (B, 1, T)
        x = self.conv(x).flatten(1)
        return self.fc(x)


print(f'Training Deep Fingerprinting (1D CNN, {N_CLASSES} classes)...')
df_model = DeepFingerprinting(n_classes=N_CLASSES)
pred = train_nn(df_model, Xseq_tr, y_tr, Xseq_te, y_te)
evaluate('Deep Fingerprinting (DF)', y_te, pred)

In [ ]:
import networkx as nx

# GraphDApp: represent flows as interaction graphs, embed with message-passing GNN.
# For CPU compatibility we extract graph-level features and use an MLP.

def graph_features(lengths):
    """Build a simple packet-interaction graph and extract graph-level features."""
    a = np.array(lengths, dtype=float)
    G = nx.DiGraph()
    for i in range(len(a) - 1):
        src = int(np.clip(a[i] // 100, 0, 19))   # discretise length into 20 bins
        dst = int(np.clip(a[i+1] // 100, 0, 19))
        if G.has_edge(src, dst):
            G[src][dst]['w'] += 1
        else:
            G.add_edge(src, dst, w=1)
    feats = [
        G.number_of_nodes(),
        G.number_of_edges(),
        nx.density(G) if G.number_of_nodes() > 1 else 0.,
        sum(d for _, d in G.in_degree())  / max(1, G.number_of_nodes()),
        sum(d for _, d in G.out_degree()) / max(1, G.number_of_nodes()),
    ]
    # degree sequence histogram (20 bins)
    in_deg  = [d for _, d in G.in_degree()]
    out_deg = [d for _, d in G.out_degree()]
    hist_in,  _ = np.histogram(in_deg,  bins=10, range=(0, 20))
    hist_out, _ = np.histogram(out_deg, bins=10, range=(0, 20))
    return np.array(feats + list(hist_in) + list(hist_out), dtype=np.float32)

print('Extracting graph features...')
Xgraph = np.array([graph_features(lengths) for lengths in
                   [all_seq[i] for i in range(len(all_seq))]])

from sklearn.preprocessing import StandardScaler
Xg_tr, Xg_te, _, _ = split(Xgraph)
sc = StandardScaler().fit(Xg_tr)
Xg_tr = sc.transform(Xg_tr)
Xg_te = sc.transform(Xg_te)

class GraphDApp(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, n_classes)
        )
    def forward(self, x):
        return self.net(x)

print(f'Training GraphDApp (Graph-feature MLP, {N_CLASSES} classes)...')
gdapp = GraphDApp(Xgraph.shape[1], N_CLASSES)
pred = train_nn(gdapp, Xg_tr, y_tr, Xg_te, y_te)
evaluate('GraphDApp', y_te, pred)

In [ ]:
class TSCRNN(nn.Module):
    """CNN + GRU hybrid for flow spatiotemporal features."""
    def __init__(self, input_len=MAX_PACKETS_PER_FLOW, n_classes=20):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool1d(2),
        )
        rnn_in = input_len // 2
        self.gru = nn.GRU(64, 128, batch_first=True)
        self.fc  = nn.Linear(128, n_classes)

    def forward(self, x):           # x: (B, T)
        x = x.unsqueeze(1)          # (B, 1, T)
        x = self.conv(x)            # (B, 64, T/2)
        x = x.permute(0, 2, 1)     # (B, T/2, 64)
        _, h = self.gru(x)
        return self.fc(h[-1])


print(f'Training TSCRNN (CNN+GRU, {N_CLASSES} classes)...')
tscrnn = TSCRNN(n_classes=N_CLASSES)
pred = train_nn(tscrnn, Xseq_tr, y_tr, Xseq_te, y_te)
evaluate('TSCRNN', y_te, pred)

In [ ]:
class Deeppacket(nn.Module):
    """Deep CNN on raw byte payload for encrypted traffic classification."""
    def __init__(self, input_len=MAX_PAYLOAD_BYTES, n_classes=20):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, stride=2, padding=2), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2), nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(8),
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 8, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        return self.fc(self.conv(x).flatten(1))


print(f'Training Deeppacket (raw-byte 1D CNN, {N_CLASSES} classes)...')
dp = Deeppacket(n_classes=N_CLASSES)
pred = train_nn(dp, Xraw_tr, y_tr, Xraw_te, y_te, epochs=15)
evaluate('Deeppacket', y_te, pred)

In [ ]:
# PERT: Transformer encoder on tokenised traffic bytes.
# We implement a lightweight version compatible with CPU training.

class PERTModel(nn.Module):
    """Lightweight Transformer encoder for traffic bytes (PERT)."""
    def __init__(self, vocab_size=256, d_model=64, nhead=4,
                 num_layers=2, seq_len=MAX_PAYLOAD_BYTES, n_classes=20):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=128,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc   = nn.Linear(d_model, n_classes)

    def forward(self, x):               # x: (B, T) float in [0,1]
        x = (x * 255).long().clamp(0, 255)
        x = self.embed(x)               # (B, T, d_model)
        x = self.transformer(x)         # (B, T, d_model)
        x = self.pool(x.permute(0,2,1)).squeeze(-1)  # (B, d_model)
        return self.fc(x)


print(f'Training PERT (Transformer on bytes, {N_CLASSES} classes)...')
# Use shorter sequences for speed on CPU
SHORT = 128
Xshort_tr = Xraw_tr[:, :SHORT]
Xshort_te = Xraw_te[:, :SHORT]

pert = PERTModel(seq_len=SHORT, n_classes=N_CLASSES)
pred = train_nn(pert, Xshort_tr, y_tr, Xshort_te, y_te, epochs=15, batch_size=32)
evaluate('PERT', y_te, pred)

In [ ]:
# ET-BERT: BERT-style bidirectional Transformer with masked token pre-training.
# Full ET-BERT pre-training requires large-scale traffic corpora and GPUs.
# Here we implement the fine-tuning stage (from random init) as a faithful
# CPU-compatible approximation of the architecture.

class ETBERTLite(nn.Module):
    """ET-BERT fine-tuning head on a lightweight bidirectional Transformer."""
    def __init__(self, vocab_size=256, d_model=128, nhead=4,
                 num_layers=3, seq_len=128, n_classes=20):
        super().__init__()
        self.tok_embed = nn.Embedding(vocab_size + 1, d_model, padding_idx=vocab_size)
        self.pos_embed = nn.Embedding(seq_len, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        encoder_layer  = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=256,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, 64), nn.Tanh(),
            nn.Linear(64, n_classes)
        )
        self.seq_len = seq_len

    def forward(self, x):               # x: (B, T) float in [0,1]
        B, T = x.shape
        T = min(T, self.seq_len)
        ids = (x[:, :T] * 255).long().clamp(0, 255)  # (B, T)
        pos = torch.arange(T, device=x.device).unsqueeze(0)  # (1, T)
        emb = self.tok_embed(ids) + self.pos_embed(pos)       # (B, T, d)
        cls = self.cls_token.expand(B, -1, -1)                # (B, 1, d)
        emb = torch.cat([cls, emb], dim=1)                    # (B, T+1, d)
        out = self.norm(self.transformer(emb))
        return self.head(out[:, 0])     # CLS token


print(f'Training ET-BERT (Bidirectional Transformer fine-tune, {N_CLASSES} classes)...')
SHORT = 128
etbert = ETBERTLite(seq_len=SHORT, n_classes=N_CLASSES)
pred = train_nn(etbert, Xraw_tr[:, :SHORT], y_tr,
                Xraw_te[:, :SHORT], y_te, epochs=20, batch_size=32, lr=5e-4)
evaluate('ET-BERT', y_te, pred)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_res = pd.DataFrame(RESULTS).T.reset_index().rename(columns={'index': 'Model'})
df_res = df_res.sort_values('F1', ascending=False).reset_index(drop=True)
df_res.to_csv('results/all_model_results.csv', index=False)

print('\n' + '='*60)
print(' RESULTS SUMMARY — USTC-TFC2016 (Macro avg)')
print('='*60)
print(tabulate(df_res, headers='keys', tablefmt='fancy_grid', showindex=False))
print('\nSaved to results/all_model_results.csv')

# ── Bar chart ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

metrics = ['F1', 'Acc']
titles  = ['Macro F1-Score', 'Accuracy']

for ax, metric, title in zip(axes, metrics, titles):
    data = df_res.sort_values(metric)
    colors = ['#2196F3' if m not in ('ET-BERT', 'PERT') else '#FF5722'
              for m in data['Model']]
    bars = ax.barh(data['Model'], data[metric], color=colors)
    ax.set_xlabel(metric)
    ax.set_title(title)
    ax.set_xlim(0, 1.05)
    for bar, val in zip(bars, data[metric]):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9)

from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(color='#2196F3', label='Statistical / DL methods'),
    Patch(color='#FF5722', label='Pre-training methods'),
], loc='lower right')

plt.suptitle('Baseline Models on USTC-TFC2016 (20 classes)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to results/comparison_chart.png')

In [ ]:
from sklearn.metrics import f1_score

# Re-run each model to collect per-class predictions for heatmap
# (We cache the last predictions from train_nn; here we demo with the sklearn models)

label_names = [k for k, _ in sorted(LABEL_MAP.items(), key=lambda x: x[1])]

models_sklearn = {
    'AppScanner': (appscanner, Xstat_te_s),
    'CUMUL':      (cumul_model, Xcumul_te_s),
    'BIND':       (bind_model, Xstat_te_s),
    'K-FP':       (kfp_model, rf_kfp.apply(Xstat_te_s)),
}

per_class = {}
for name, (mdl, Xte) in models_sklearn.items():
    preds = mdl.predict(Xte)
    f1s = f1_score(y_te, preds, average=None, labels=list(range(N_CLASSES)), zero_division=0)
    per_class[name] = f1s

df_pc = pd.DataFrame(per_class, index=label_names[:N_CLASSES])

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df_pc, annot=True, fmt='.2f', cmap='YlOrRd',
            vmin=0, vmax=1, ax=ax, linewidths=0.3)
ax.set_title('Per-Class F1-Score Heatmap (Statistical Models)', fontsize=13)
ax.set_xlabel('Model')
ax.set_ylabel('Traffic Class')
plt.tight_layout()
plt.savefig('results/per_class_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved to results/per_class_heatmap.png')

In [ ]:
import pickle, torch

os.makedirs('saved_models', exist_ok=True)

# sklearn models
for name, obj in [
    ('appscanner',   appscanner),
    ('cumul',        cumul_model),
    ('bind',         bind_model),
    ('kfp_rf',       rf_kfp),
    ('kfp_knn',      kfp_model),
    ('flowprint_km', kmeans),
    ('scaler_stat',  scaler_stat),
    ('scaler_cumul', scaler_cumul),
    ('label_map',    LABEL_MAP),
]:
    with open(f'saved_models/{name}.pkl', 'wb') as f:
        pickle.dump(obj, f)

# PyTorch models
for name, mdl in [
    ('fsnet',   fsnet),
    ('df',      df_model),
    ('gdapp',   gdapp),
    ('tscrnn',  tscrnn),
    ('deeppacket', dp),
    ('pert',    pert),
    ('etbert',  etbert),
]:
    torch.save(mdl.state_dict(), f'saved_models/{name}.pt')

print('All models saved to saved_models/')

In [ ]:
def predict_pcap(pcap_path, model_name='AppScanner'):
    """
    Predict traffic class for all flows in a new PCAP file.
    model_name: one of 'AppScanner', 'CUMUL', 'BIND', 'K-FP', 'FS-Net',
                'Deep Fingerprinting (DF)', 'TSCRNN', 'Deeppacket',
                'PERT', 'ET-BERT'
    """
    flows = extract_flows(pcap_path, max_flows=50)
    if not flows:
        print('No flows extracted.'); return

    inv_map = {v: k for k, v in LABEL_MAP.items()}

    for i, f in enumerate(flows[:5]):
        if model_name == 'AppScanner':
            feat = scaler_stat.transform([stat_features(f['lengths'])])
            pred_id = appscanner.predict(feat)[0]
        elif model_name == 'CUMUL':
            feat = scaler_cumul.transform([cumul_features(f['lengths'])])
            pred_id = cumul_model.predict(feat)[0]
        elif model_name == 'BIND':
            feat = scaler_stat.transform([stat_features(f['lengths'])])
            pred_id = bind_model.predict(feat)[0]
        elif model_name == 'K-FP':
            feat = scaler_stat.transform([stat_features(f['lengths'])])
            leaf = rf_kfp.apply(feat)
            pred_id = kfp_model.predict(leaf)[0]
        elif model_name == 'FS-Net':
            x = torch.FloatTensor([pad_sequence(f['lengths'])])
            fsnet.eval()
            with torch.no_grad(): pred_id = fsnet(x).argmax(1).item()
        elif model_name == 'Deep Fingerprinting (DF)':
            x = torch.FloatTensor([pad_sequence(f['lengths'])])
            df_model.eval()
            with torch.no_grad(): pred_id = df_model(x).argmax(1).item()
        elif model_name == 'Deeppacket':
            x = torch.FloatTensor([raw_features(f['raw'])])
            dp.eval()
            with torch.no_grad(): pred_id = dp(x).argmax(1).item()
        elif model_name == 'PERT':
            x = torch.FloatTensor([raw_features(f['raw'])[:SHORT]])
            pert.eval()
            with torch.no_grad(): pred_id = pert(x).argmax(1).item()
        elif model_name == 'ET-BERT':
            x = torch.FloatTensor([raw_features(f['raw'])[:SHORT]])
            etbert.eval()
            with torch.no_grad(): pred_id = etbert(x).argmax(1).item()
        else:
            print(f'Unknown model: {model_name}'); return

        print(f'  Flow {i+1}: predicted = {inv_map.get(pred_id, pred_id)}')


# Example usage — uncomment and set your pcap path:
# predict_pcap('data/Benign/BitTorrent.pcap', model_name='AppScanner')

print('predict_pcap() is ready to use.')